# Databricks Host and Token
### The Personal Access Token is used to authenticate your SDK calls. It's a secret key, so treat it like a password!
1. In your Databricks workspace, click your username in the top-right corner.
2. Select Settings.
3. Click the Developer tab.
4. Next to Access tokens, click Manage.
5. Click the Generate new token button.
6. Enter a Comment (e.g., "SDK Access") and set a Lifetime for the token (the recommended practice is to set an expiration date).
7. Click Generate.
8. Immediately copy the displayed token. This is the only time Databricks will show you the token. If you lose it, you'll have to generate a new one.
9. Add them to the databricksconfig file generated by the cli install.

In [8]:
import os
from databricks.sdk import WorkspaceClient
import datetime as dt

# https://github.com/AgDMALabs-Public/ag-vision-dataops
from ag_vision.mobile.ingest import MobileImageIngest

# Connect to Roboflow

In [2]:
w = WorkspaceClient(profile='agpile')  # set your profile
w.config.host

'https://dbc-0f3d94f2-e27b.cloud.databricks.com'

# Define Local and Databricks Variables.

In [9]:
DATA_LOCAL_PATH = "" # EX: '/Users/danielwilliams/Documents/project data/board data/'
DB_PROJECT_DIR = "" # EX: '/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data'

COLLECTION_DATE = dt.datetime.now().strftime('%Y-%m-%d')
TRIAL = 'test'
SITE = 'California-labs'
FIELD = 'home'
LOCATION = 'home_1'

TASK = 'grain_imaging'
PROTOCOL = 'corn_blue_board'

YEAR = 2025
COUNTRY = 'IND'
CROP = 'maize'
TIME_OF_YEAR = 'spring'

# List out the files you want to upload

In [10]:
files = os.listdir(DATA_LOCAL_PATH)
files = [x for x in files if '.jpeg' in x]
files = [DATA_LOCAL_PATH + x for x in files]
files

['/Users/danielwilliams/Documents/project data/board data/IMG_3249.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3253.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3252.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3248.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3255.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3259.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3258.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3254.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3257.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3260.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3256.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3251.jpeg',
 '/Users/danielwilliams/Documents/project data/board data/IMG_3250.jpeg']

In [11]:
# create an empty MobileImageIngest object.
ingest = MobileImageIngest(platform='local',
                           cloud_client=w,
                           cloud_bucket=None,
                           ingest_df=None)

In [12]:
ingest.generate_ingest_df(file_list=files)


In [13]:
ingest.ingest_df.head()


,src_path,img_ext,project_dir,site,trial,year,country,crop,time_of_year,season,field,location,task,protocol,collection_date,id,plot_id,event_type
0,/Users/danielwilliams/Documents/project data/b...,.jpeg,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,/Users/danielwilliams/Documents/project data/b...,.jpeg,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,/Users/danielwilliams/Documents/project data/b...,.jpeg,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,/Users/danielwilliams/Documents/project data/b...,.jpeg,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,/Users/danielwilliams/Documents/project data/b...,.jpeg,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [14]:
# generate unique ID's for each of the images. This method used UUID4()
ingest.generate_unique_image_ids()

In [16]:
# these are all the columns that you need to upload mobile scouting data to Fairgorunds and follow the Data Architecture Laid out in ag_vision.
ingest.ingest_df['project_dir'] = DB_PROJECT_DIR
ingest.ingest_df['site'] = SITE
ingest.ingest_df['trial'] = TRIAL
ingest.ingest_df['year'] = YEAR
ingest.ingest_df['country'] = COUNTRY
ingest.ingest_df['crop'] = CROP
ingest.ingest_df['time_of_year'] = TIME_OF_YEAR
ingest.ingest_df['field'] = FIELD
ingest.ingest_df['location'] = LOCATION
ingest.ingest_df['task'] = TASK
ingest.ingest_df['protocol'] = PROTOCOL
ingest.ingest_df['event_type'] = 'scouting'
ingest.ingest_df['collection_date'] = COLLECTION_DATE
ingest.ingest_df['plot_id'] = 'none' # if you want to upload event_type of trial you will need plot ID's for each image.

In [17]:
ingest.ingest_df.head()


,src_path,img_ext,project_dir,site,trial,year,country,crop,time_of_year,season,field,location,task,protocol,collection_date,id,plot_id,event_type
0,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,None,home,home_1,grain_imaging,corn_blue_board,2026-01-20,eb676d60-dab7-4c73-9ebc-51594875987a,none,scouting
1,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,None,home,home_1,grain_imaging,corn_blue_board,2026-01-20,901fbc26-7455-4556-9d98-2d8f22dbba4d,none,scouting
2,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,None,home,home_1,grain_imaging,corn_blue_board,2026-01-20,87ea7ec6-fe21-4271-9848-930c12800827,none,scouting
3,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,None,home,home_1,grain_imaging,corn_blue_board,2026-01-20,5be4b5e7-4b35-41f7-a876-b76c44be570d,none,scouting
4,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,None,home,home_1,grain_imaging,corn_blue_board,2026-01-20,c1629596-8b89-475f-b5e1-8785026a3d18,none,scouting


In [18]:
# From the year, country, crop, and time of year we can generate a season code.
ingest.add_season_column()

100%|██████████| 13/13 [00:00<00:00, 3398.53it/s]


In [19]:
# Validate that all the columns are there and for some columns valdiate what is in them.
ingest.validate_ingest_df()

In [20]:
# generate the dst_path from all the data in the column of ingest_df.
ingest.generate_dst_path_name()

In [21]:
ingest.ingest_df.head()


,src_path,img_ext,project_dir,site,trial,year,country,crop,time_of_year,season,field,location,task,protocol,collection_date,id,plot_id,event_type,dst_path
0,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,2025:ind:maize:spring,home,home_1,grain_imaging,corn_blue_board,2026-01-20,eb676d60-dab7-4c73-9ebc-51594875987a,none,scouting,/Volumes/ue1_prod_catalog_119738067017277/tier...
1,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,2025:ind:maize:spring,home,home_1,grain_imaging,corn_blue_board,2026-01-20,901fbc26-7455-4556-9d98-2d8f22dbba4d,none,scouting,/Volumes/ue1_prod_catalog_119738067017277/tier...
2,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,2025:ind:maize:spring,home,home_1,grain_imaging,corn_blue_board,2026-01-20,87ea7ec6-fe21-4271-9848-930c12800827,none,scouting,/Volumes/ue1_prod_catalog_119738067017277/tier...
3,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,2025:ind:maize:spring,home,home_1,grain_imaging,corn_blue_board,2026-01-20,5be4b5e7-4b35-41f7-a876-b76c44be570d,none,scouting,/Volumes/ue1_prod_catalog_119738067017277/tier...
4,/Users/danielwilliams/Documents/project data/b...,.jpeg,/Volumes/ue1_prod_catalog_119738067017277/tier...,California-labs,test,2025,IND,maize,spring,2025:ind:maize:spring,home,home_1,grain_imaging,corn_blue_board,2026-01-20,c1629596-8b89-475f-b5e1-8785026a3d18,none,scouting,/Volumes/ue1_prod_catalog_119738067017277/tier...


In [24]:
ingest.ingest_df['dst_path'][0]

'/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data/california-labs/test/2025:ind:maize:spring/home/home_1/grain_imaging/images/corn_blue_board/2026-01-20/eb676d60-dab7-4c73-9ebc-51594875987a.jpeg'

In [23]:
# this uploads the data to the cloud. If files failed they will be stored in the failed_upload list.
ingest.upload_local_data_to_db()

  0%|          | 0/13 [00:00<?, ?it/s]
Uploading:   0%|          | 0.00/5.03M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/5.33M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/7.92M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/5.42M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.61M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/4.96M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.74M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/8.16M [00:00<?, ?B/s]

Uploading:   0%|          | 0.00/4.73M [00:00<?, ?B/s]
100%|██████████| 13/13 [00:00<00:00, 626.66it/s]
